# CBD Adapter — Unity / VRM

**`adapter/` role of the [Common Behavior Data](https://github.com/Koichi3333/common-behavior-data) project.**
Target: **Unity / VRM 1.0** (`motion.vrma`).

This notebook reads the canonical dataset written by
`cbd_generator_video_to_cbd.ipynb` and maps the same bone rotations that the
MuJoCo adapter uses onto VRM humanoid bones, written as glTF (GLB) with the
`VRMC_vrm_animation` extension. Any VRM 1.0 avatar can play it, and Unity does
no inference and no motion computation — Load VRM, Load VRMA, Play.

```text
cbd_dataset.zip ─▶ [ THIS NOTEBOOK ] ─▶ output/03_unity_vrm/motion.vrma
 (canonical CBD)     wxyz ➜ xyzw                          object_trajectory_unity.json
                                                          ObjectTrajectoryPlayer.cs
                                                          README_UNITY.md
                                                          adapter_report.json
```

## Adapter rules honoured here

1. **Reads canonical data** — `timeline/frames.jsonl` and `manifest.json` only.
   It does *not* read the MuJoCo adapter's output, even though both project
   the same rotations.
2. **Converts at its boundary** — canonical is already Y-up right-handed with
   the person facing +Z, which is glTF's convention, so the conversion is the
   quaternion component order (`wxyz` → `xyzw`) plus the VRM rest hip height
3. **Model and motion stay separate** — the avatar (`avatar.vrm`) is yours;
   this notebook only writes the motion
4. **Does not promote candidates** — no interaction candidate is baked into
   the animation
5. **Reports what it could not represent** in `adapter_report.json`
   (facial expressions, per-joint finger angles, object rotation, …)
6. **Preserves provenance** — every frame of
   `object_trajectory_unity.json` keeps its `source` field

## Input

Any one of these, tried in order — no editing needed:

1. the dataset already unpacked in this runtime (the generator just ran here)
2. `/content/cbd_dataset.zip` (`colab upload`)
3. the Colab upload widget (UI only)

## How to run — colab CLI

```bash
colab exec -s cbd -f cbd_adapter_unity_vrm.ipynb --timeout 900
colab download -s cbd \
  /content/human_behavior_demo_2_0/vrm_adapter_output.zip ./vrm_adapter_output.zip
```

Nothing here needs a GPU: this adapter writes files, it does not render.
Playback happens in Unity on your own machine — see the guide at the end.


In [ ]:
# @title [A1] Environment setup
# =====================================================================
# Unity / VRM adapter 1/5.
# This adapter writes a glTF container by hand, so there is nothing to
# install: numpy and pandas are enough, and no GPU is involved.
# =====================================================================
import json
import math
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Run mode: Colab UI or headless (colab CLI / Run all / papermill)
# ---------------------------------------------------------------
# The two entry points differ in exactly one way that matters here: whether
# the kernel accepts stdin. `colab exec` calls execute_code() without
# allow_stdin, so an upload widget would hang until the run times out, and an
# embedded base64 video player would dump megabytes into the log. Detect the
# mode once, then degrade gracefully instead of blocking.
def _stdin_available():
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or not hasattr(shell, "kernel"):
            return False
        return bool(getattr(shell.kernel, "_allow_stdin", False))
    except Exception:  # noqa: BLE001
        return False


IS_COLAB_UI = _stdin_available()
SHOW_MEDIA = IS_COLAB_UI      # embed videos / images only in the browser UI
RUN_MODE = "colab-ui" if IS_COLAB_UI else "headless (colab CLI / Run all)"
print(f"Run mode: {RUN_MODE}")
print("numpy:", np.__version__, "| pandas:", pd.__version__)


In [ ]:
# @title [A2] Load the Common Behavior Data
# ---------------------------------------------------------------
# Locate the Common Behavior Data
# ---------------------------------------------------------------
# Resolution order, so the same cell works from either entry point:
#   1. an already-unpacked dataset in this runtime (the generator just ran)
#   2. cbd_dataset.zip sitting on the VM (`colab upload`)
#   3. the upload widget -- Colab UI only
#   4. otherwise: stop with the exact command needed, instead of hanging
CBD_DIR_OVERRIDE = ""      # optional: a directory containing 04_behavior_dataset/

PROJECT_DIR = Path("/content/human_behavior_demo_2_0")
OUTPUT_DIR = PROJECT_DIR / "output"
WORK_DIR = PROJECT_DIR / "_work"
ZIP_CANDIDATES = ["/content/cbd_dataset.zip", "cbd_dataset.zip",
                  str(PROJECT_DIR / "cbd_dataset.zip")]


def _find_dataset(root):
    """Return the 04_behavior_dataset directory below root, if any."""
    root = Path(root)
    direct = root / "output/04_behavior_dataset/manifest.json"
    if direct.exists():
        return direct.parent
    if (root / "manifest.json").exists() and root.name == "04_behavior_dataset":
        return root
    hit = next(iter(sorted(root.glob("**/04_behavior_dataset/manifest.json"))), None)
    return hit.parent if hit else None


def _unpack(zip_path):
    print(f"Unpacking {zip_path} -> {PROJECT_DIR}")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)


DATASET_DIR = _find_dataset(CBD_DIR_OVERRIDE) if CBD_DIR_OVERRIDE else None
if DATASET_DIR is None:
    DATASET_DIR = _find_dataset(PROJECT_DIR)
if DATASET_DIR is None:
    _zip = next((p for p in ZIP_CANDIDATES if Path(p).exists()), None)
    if _zip is None and IS_COLAB_UI:
        print("No Common Behavior Data on this VM. Upload cbd_dataset.zip "
              "(produced by cbd_generator_video_to_cbd.ipynb).")
        from google.colab import files
        _uploaded = files.upload()
        _zip = "/content/" + next(iter(_uploaded))
    if _zip is None:
        raise RuntimeError(
            "No Common Behavior Data on this VM and this is a headless run, "
            "so no upload widget can be opened.\n"
            "Run the generator on this session first, or upload its output:\n"
            "    colab upload -s <session> ./cbd_dataset.zip "
            "/content/cbd_dataset.zip")
    _unpack(_zip)
    DATASET_DIR = _find_dataset(PROJECT_DIR)
if DATASET_DIR is None:
    raise RuntimeError("cbd_dataset.zip does not contain 04_behavior_dataset/")

OUTPUT_DIR = DATASET_DIR.parent
# Normal layout: <project>/output/04_behavior_dataset. A flat bundle (the
# dataset directly under the extract root) is accepted too -- then the extract
# root doubles as the project root, so nothing is written outside it.
PROJECT_DIR = OUTPUT_DIR.parent if OUTPUT_DIR.name == "output" else OUTPUT_DIR
WORK_DIR = PROJECT_DIR / "_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_VIDEO = PROJECT_DIR / "source/source_video.mp4"
if not SOURCE_VIDEO.exists():
    SOURCE_VIDEO = next(iter(sorted(PROJECT_DIR.glob("**/source_video.mp4"))),
                        SOURCE_VIDEO)

print("Common Behavior Data:", DATASET_DIR)

# ---------------------------------------------------------------
# Read the canonical timeline
# ---------------------------------------------------------------
# Everything below comes from files. No MediaPipe, no re-computation: that is
# the whole point of the adapter boundary.
MANIFEST = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
SUMMARY = json.loads(
    (DATASET_DIR / "behavior_summary.json").read_text(encoding="utf-8"))
GENERATOR_CONFIG = {}
if (PROJECT_DIR / "config.json").exists():
    GENERATOR_CONFIG = json.loads(
        (PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

FRAMES = [json.loads(line) for line
          in (DATASET_DIR / "timeline/frames.jsonl").read_text(
              encoding="utf-8").splitlines() if line.strip()]
NUM_FRAMES = len(FRAMES)
if NUM_FRAMES == 0:
    raise RuntimeError("timeline/frames.jsonl is empty")
FPS = float(MANIFEST["fps"])
DT = 1.0 / FPS
TIMESTAMPS = np.array([f["timestamp_sec"] for f in FRAMES], dtype=np.float32)
SOURCE_FRAME_INDEX = [f["source_frame_index"] for f in FRAMES]

BONE_ORDER = (MANIFEST.get("bone_order")
              or list(FRAMES[0]["human"]["bone_rotations_xyzw"]))
FINGER_ORDER = MANIFEST.get("finger_order",
                            ["thumb", "index", "middle", "ring", "little"])

# Bone rotations: stored xyzw in the file, used wxyz by the quaternion helpers
BONE_ROT = {}
for bone in BONE_ORDER:
    xyzw = np.array([f["human"]["bone_rotations_xyzw"][bone] for f in FRAMES],
                    dtype=np.float64)
    BONE_ROT[bone] = np.column_stack([xyzw[:, 3], xyzw[:, 0], xyzw[:, 1],
                                      xyzw[:, 2]])
HIPS_POS = np.array([f["human"]["hips_position"] for f in FRAMES],
                    dtype=np.float64)

# Finger curls [rad]. The timeline stores null on frames where the hand was
# not detected; fill from the nearest valid frame so a renderer does not snap
# the hand open, and record how many frames that affected.
ADAPTER_NOTES = []       # what this adapter could not represent, or had to fill
CURLS, HAND_PRESENT, CURLS_FILLED = {}, {}, {}
for _side in ["left", "right"]:
    raw = [f["human"]["finger_curls_rad"].get(_side) for f in FRAMES]
    present = np.array([r is not None for r in raw])
    values = np.zeros((NUM_FRAMES, len(FINGER_ORDER)))
    if present.any():
        valid_idx = np.where(present)[0]
        for fi in range(NUM_FRAMES):
            src = fi if present[fi] else valid_idx[np.argmin(np.abs(valid_idx - fi))]
            values[fi] = raw[src]
    CURLS[_side.capitalize()] = values
    HAND_PRESENT[_side.capitalize()] = present
    CURLS_FILLED[_side.capitalize()] = int((~present).sum()) if present.any() else 0
    if present.any() and CURLS_FILLED[_side.capitalize()]:
        ADAPTER_NOTES.append(
            f"finger curls: {CURLS_FILLED[_side.capitalize()]}/{NUM_FRAMES} "
            f"{_side} frames had no hand detection and were filled from the "
            "nearest detected frame")
    if not present.any():
        ADAPTER_NOTES.append(f"finger curls: no {_side} hand was ever detected")

PHASE = [f["phase"] for f in FRAMES]
INTERACTIONS = [f["interactions"] for f in FRAMES]
CAPTIONS = [f.get("caption") for f in FRAMES]

# Primary object. "primary_object" is defined on every frame; datasets written
# before that field existed are read back from the objects[] list instead.
PRIMARY_LABEL = (MANIFEST.get("primary", {}).get("object_label")
                 or (SUMMARY.get("primary_object") or {}).get("label"))
PRIMARY_TRACK_ID = (MANIFEST.get("primary", {}).get("object_track_id")
                    or (SUMMARY.get("primary_object") or {}).get("track_id"))
PRIMARY_HAND = (MANIFEST.get("primary", {}).get("hand")
                or SUMMARY.get("primary_hand") or "right")

OBJ_PROXY, OBJ_SOURCE = None, ["none"] * NUM_FRAMES
if PRIMARY_TRACK_ID:
    positions, sources, missing = [], [], 0
    for f in FRAMES:
        block = f.get("primary_object")
        if block is None:
            block = next((o for o in f["objects"]
                          if o.get("track_id") == PRIMARY_TRACK_ID
                          and "proxy_canonical" in o), None)
        if block and block.get("proxy_canonical"):
            positions.append(block["proxy_canonical"])
            sources.append(block.get("position_source", "unknown"))
        else:
            positions.append(positions[-1] if positions else [0.0, 0.9, 0.3])
            sources.append("carried_forward_by_adapter")
            missing += 1
    OBJ_PROXY = np.array(positions, dtype=np.float64)
    OBJ_SOURCE = sources
    if missing:
        ADAPTER_NOTES.append(
            f"object proxy: {missing}/{NUM_FRAMES} frames had no canonical "
            "position in the dataset and were carried forward by this adapter")

print(f"frames={NUM_FRAMES}  fps={FPS:.2f}  duration={TIMESTAMPS[-1]:.2f}s")
print(f"task={SUMMARY.get('task')}  primary_hand={PRIMARY_HAND}  "
      f"primary_object={PRIMARY_LABEL or 'none'}")
print("events:", SUMMARY.get("events") or "none")
print("captions:", sum(1 for c in CAPTIONS if c), "frames carry a caption")
for note in ADAPTER_NOTES:
    print("  note:", note)


def write_adapter_report(path, adapter, target, reads, unrepresented):
    """Rule 5: say what could not be represented, rather than approximating
    it silently. Rule 6: say where every derived value came from."""
    report = {
        "adapter": adapter,
        "target": target,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "source_dataset": {
            "manifest": str(Path(DATASET_DIR) / "manifest.json"),
            "dataset_version": MANIFEST.get("dataset_version"),
            "frame_count": NUM_FRAMES,
            "fps": FPS,
        },
        "reads": reads,
        "filled_or_interpolated": list(ADAPTER_NOTES),
        "not_represented": unrepresented,
        "promoted_candidates": False,
    }
    Path(path).write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Wrote:", path)
    return report
print("bones:", len(BONE_ROT), "| object proxy:",
      "yes" if OBJ_PROXY is not None else "no")


In [ ]:
# @title [A3] Export motion.vrma (VRM Animation)
# =====================================================================
# Unity / VRM adapter 3/5.
# The same canonical bone rotations, this time mapped onto VRM humanoid bones
# and written as glTF (GLB) + the VRMC_vrm_animation extension -- playable in
# Unity on any VRM 1.0 avatar.
# Coordinate conversion at this adapter's boundary: canonical is already Y-up
# right-handed with the person facing +Z, which is glTF's own convention, so
# the only change is the quaternion component order (wxyz -> xyzw).
# =====================================================================
import struct

VRM_CFG = {
    "export_fingers": True,
    "rest_hip_height": 0.95,   # VRM rest pose hip height; the dataset stores
                               # hips_position as an offset around the origin
}

UNITY_DIR = OUTPUT_DIR / "03_unity_vrm"
UNITY_DIR.mkdir(parents=True, exist_ok=True)

# ---- Common bone -> VRM humanoid bone mapping ----
COMMON_TO_VRM = {
    "hips": "hips", "spine": "spine", "chest": "chest",
    "neck": "neck", "head": "head",
    "left_upper_arm": "leftUpperArm", "left_lower_arm": "leftLowerArm",
    "left_hand": "leftHand",
    "right_upper_arm": "rightUpperArm", "right_lower_arm": "rightLowerArm",
    "right_hand": "rightHand",
    "left_upper_leg": "leftUpperLeg", "left_lower_leg": "leftLowerLeg",
    "left_foot": "leftFoot",
    "right_upper_leg": "rightUpperLeg", "right_lower_leg": "rightLowerLeg",
    "right_foot": "rightFoot",
}

# ---- VRMA skeleton (T-pose, glTF Y-up, person facing +Z) ----
VRM_NODE_DEFS = [
    ("hips", None, (0.0, 0.95, 0.0)),
    ("spine", "hips", (0.0, 0.10, 0.0)),
    ("chest", "spine", (0.0, 0.12, 0.0)),
    ("neck", "chest", (0.0, 0.20, 0.0)),
    ("head", "neck", (0.0, 0.06, 0.0)),
    ("leftUpperArm", "chest", (0.17, 0.14, 0.0)),
    ("leftLowerArm", "leftUpperArm", (0.26, 0.0, 0.0)),
    ("leftHand", "leftLowerArm", (0.25, 0.0, 0.0)),
    ("rightUpperArm", "chest", (-0.17, 0.14, 0.0)),
    ("rightLowerArm", "rightUpperArm", (-0.26, 0.0, 0.0)),
    ("rightHand", "rightLowerArm", (-0.25, 0.0, 0.0)),
    ("leftUpperLeg", "hips", (0.09, -0.05, 0.0)),
    ("leftLowerLeg", "leftUpperLeg", (0.0, -0.42, 0.0)),
    ("leftFoot", "leftLowerLeg", (0.0, -0.40, 0.0)),
    ("rightUpperLeg", "hips", (-0.09, -0.05, 0.0)),
    ("rightLowerLeg", "rightUpperLeg", (0.0, -0.42, 0.0)),
    ("rightFoot", "rightLowerLeg", (0.0, -0.40, 0.0)),
]

# ---- Finger bones (only where the mapping is stable enough to trust) ----
FINGER_SEGMENTS = {
    "Thumb": ["Metacarpal", "Proximal", "Distal"],
    "Index": ["Proximal", "Intermediate", "Distal"],
    "Middle": ["Proximal", "Intermediate", "Distal"],
    "Ring": ["Proximal", "Intermediate", "Distal"],
    "Little": ["Proximal", "Intermediate", "Distal"],
}
FINGER_Z_OFFSET = {"Thumb": 0.030, "Index": 0.020, "Middle": 0.0,
                   "Ring": -0.018, "Little": -0.034}
EXPORT_FINGERS = (VRM_CFG["export_fingers"]
                  and (HAND_PRESENT["Left"].any() or HAND_PRESENT["Right"].any()))
if EXPORT_FINGERS:
    for side, sign in [("left", 1.0), ("right", -1.0)]:
        hand_node = f"{side}Hand"
        for finger, segments in FINGER_SEGMENTS.items():
            parent = hand_node
            for si, segment in enumerate(segments):
                node_name = f"{side}{finger}{segment}"
                if si == 0:
                    translation = (sign * 0.035, 0.0, FINGER_Z_OFFSET[finger])
                else:
                    translation = (sign * 0.030, 0.0, 0.0)
                VRM_NODE_DEFS.append((node_name, parent, translation))
                parent = node_name

NODE_INDEX = {name: i for i, (name, _, _) in enumerate(VRM_NODE_DEFS)}

# ---- Assemble the animation tracks ----
times = TIMESTAMPS.astype(np.float32)


def axis_angle_quat_xyzw(axis, angle):
    axis = np.asarray(axis, float)
    s = math.sin(angle / 2)
    return np.array([axis[0]*s, axis[1]*s, axis[2]*s, math.cos(angle / 2)],
                    dtype=np.float32)


rotation_tracks = {}   # VRM node name -> (F, 4) quaternions, xyzw
for common, vrm in COMMON_TO_VRM.items():
    q = BONE_ROT[common]  # (F, 4) wxyz, canonical local -- already glTF space
    rotation_tracks[vrm] = np.stack(
        [q[:, 1], q[:, 2], q[:, 3], q[:, 0]], axis=1).astype(np.float32)

if EXPORT_FINGERS:
    for side_key, side_name, axis in [("Left", "left", (0, 0, -1.0)),
                                      ("Right", "right", (0, 0, 1.0))]:
        for finger_i, finger in enumerate(["Thumb", "Index", "Middle",
                                           "Ring", "Little"]):
            curls = CURLS[side_key][:, finger_i]  # rad, whole finger
            segments = FINGER_SEGMENTS[finger]
            per_segment = np.clip(curls / len(segments), 0.0, 1.6)
            track = np.stack([axis_angle_quat_xyzw(axis, a) for a in per_segment])
            for segment in segments:
                rotation_tracks[f"{side_name}{finger}{segment}"] = track

hips_translation = np.stack([
    HIPS_POS[:, 0], VRM_CFG["rest_hip_height"] + HIPS_POS[:, 1],
    HIPS_POS[:, 2]], axis=1).astype(np.float32)

# ---- Write the GLB container ----
binary_blob = bytearray()
buffer_views = []
accessors = []


def add_accessor(array, accessor_type, with_minmax=False):
    global binary_blob
    data = array.astype(np.float32).tobytes()
    offset = len(binary_blob)
    binary_blob += data
    while len(binary_blob) % 4:
        binary_blob += b"\x00"
    buffer_views.append({"buffer": 0, "byteOffset": offset,
                         "byteLength": len(data)})
    accessor = {"bufferView": len(buffer_views) - 1, "componentType": 5126,
                "count": len(array), "type": accessor_type}
    if with_minmax:
        flat = array.reshape(len(array), -1)
        accessor["min"] = [float(v) for v in flat.min(axis=0)]
        accessor["max"] = [float(v) for v in flat.max(axis=0)]
    accessors.append(accessor)
    return len(accessors) - 1


time_accessor = add_accessor(times.reshape(-1, 1), "SCALAR", with_minmax=True)

samplers, channels = [], []
for node_name, track in rotation_tracks.items():
    output_accessor = add_accessor(track, "VEC4")
    samplers.append({"input": time_accessor, "output": output_accessor,
                     "interpolation": "LINEAR"})
    channels.append({"sampler": len(samplers) - 1,
                     "target": {"node": NODE_INDEX[node_name],
                                "path": "rotation"}})
translation_accessor = add_accessor(hips_translation, "VEC3")
samplers.append({"input": time_accessor, "output": translation_accessor,
                 "interpolation": "LINEAR"})
channels.append({"sampler": len(samplers) - 1,
                 "target": {"node": NODE_INDEX["hips"], "path": "translation"}})

nodes = []
for name, parent, translation in VRM_NODE_DEFS:
    nodes.append({"name": name, "translation": list(translation)})
for i, (name, parent, _) in enumerate(VRM_NODE_DEFS):
    if parent is not None:
        nodes[NODE_INDEX[parent]].setdefault("children", []).append(i)

# Register only names that are valid VRM humanoid bones
valid_vrm_bones = set(COMMON_TO_VRM.values())
if EXPORT_FINGERS:
    for side in ["left", "right"]:
        for finger, segments in FINGER_SEGMENTS.items():
            for segment in segments:
                valid_vrm_bones.add(f"{side}{finger}{segment}")
human_bones = {name: {"node": NODE_INDEX[name]}
               for name in NODE_INDEX if name in valid_vrm_bones}

gltf_json = {
    "asset": {"version": "2.0",
              "generator": "human_behavior_demo_2_0 vrm_adapter"},
    "extensionsUsed": ["VRMC_vrm_animation"],
    "extensions": {"VRMC_vrm_animation": {
        "specVersion": "1.0",
        "humanoid": {"humanBones": human_bones}}},
    "scene": 0,
    "scenes": [{"nodes": [NODE_INDEX["hips"]]}],
    "nodes": nodes,
    "animations": [{"name": "human_behavior_motion",
                    "samplers": samplers, "channels": channels}],
    "buffers": [{"byteLength": len(binary_blob)}],
    "bufferViews": buffer_views,
    "accessors": accessors,
}

json_bytes = json.dumps(gltf_json, separators=(",", ":")).encode("utf-8")
while len(json_bytes) % 4:
    json_bytes += b" "
bin_bytes = bytes(binary_blob)
while len(bin_bytes) % 4:
    bin_bytes += b"\x00"

VRMA_PATH = UNITY_DIR / "motion.vrma"
total_length = 12 + 8 + len(json_bytes) + 8 + len(bin_bytes)
with open(VRMA_PATH, "wb") as fp:
    fp.write(struct.pack("<III", 0x46546C67, 2, total_length))
    fp.write(struct.pack("<II", len(json_bytes), 0x4E4F534A))  # 'JSON'
    fp.write(json_bytes)
    fp.write(struct.pack("<II", len(bin_bytes), 0x004E4942))   # 'BIN'
    fp.write(bin_bytes)

# adapters/vrm_bone_rotations.csv
# adapters/ is the one place inside the dataset an adapter may write: it holds
# derived, engine-specific views, never canonical data.
(DATASET_DIR / "adapters").mkdir(parents=True, exist_ok=True)
rows = []
for fi in range(NUM_FRAMES):
    for common, vrm in COMMON_TO_VRM.items():
        x, y, z, w = rotation_tracks[vrm][fi]
        rows.append([fi, FRAMES[fi]["timestamp_ms"], vrm,
                     round(float(x), 5), round(float(y), 5),
                     round(float(z), 5), round(float(w), 5)])
pd.DataFrame(rows, columns=["frame", "timestamp_ms", "vrm_bone",
                            "rot_x", "rot_y", "rot_z", "rot_w"]).to_csv(
    DATASET_DIR / "adapters/vrm_bone_rotations.csv", index=False)


print(f"Wrote: {VRMA_PATH}  ({VRMA_PATH.stat().st_size/1024:.0f} KB, "
      f"{len(rotation_tracks)} bone tracks, fingers={EXPORT_FINGERS})")


In [ ]:
# @title [A4] Object trajectory for Unity + playback guide
# =====================================================================
# Unity / VRM adapter 4/5.
# The object trajectory is part of the Common Behavior Data too, so it
# replays the same way the human motion does. VRMA only covers humanoids, so
# the object travels via a small JSON + C# script instead.
# =====================================================================
if OBJ_PROXY is not None and PRIMARY_LABEL:
    CYLINDER_LABELS = {"cup", "bottle", "wine glass", "vase", "bowl"}
    SPHERE_LABELS = {"sports ball", "ball", "apple", "orange", "frisbee"}
    obj_label = PRIMARY_LABEL
    obj_shape = ("cylinder" if obj_label in CYLINDER_LABELS
                 else "sphere" if obj_label in SPHERE_LABELS else "cube")
    trajectory_unity = {
        "label": obj_label,
        "shape": obj_shape,
        "fps": FPS,
        "size": 0.12,
        "coordinate_note": ("canonical Y-up right-handed; the C# player "
                            "negates X to match UniVRM's VRM 1.0 import"),
        "frames": [{"t": round(float(TIMESTAMPS[fi]), 4),
                    "x": round(float(OBJ_PROXY[fi][0]), 4),
                    "y": round(float(OBJ_PROXY[fi][1]), 4),
                    "z": round(float(OBJ_PROXY[fi][2]), 4),
                    "source": OBJ_SOURCE[fi]}
                   for fi in range(NUM_FRAMES)],
    }
    (UNITY_DIR / "object_trajectory_unity.json").write_text(
        json.dumps(trajectory_unity, indent=1), encoding="utf-8")

    OBJECT_PLAYER_CS = r'''using System;
using UnityEngine;

// Kinematic replay of a Common Behavior Data object trajectory in Unity.
// Usage: drop object_trajectory_unity.json and this file into Assets/, add
// this component to an empty GameObject, and assign trajectoryJson.
// Press Space during playback to re-sync with the avatar's loop start.
public class ObjectTrajectoryPlayer : MonoBehaviour
{
    [Serializable] public class TrajectoryFrame
    { public float t; public float x; public float y; public float z; public string source; }
    [Serializable] public class ObjectTrajectory
    { public string label; public string shape; public float fps; public float size;
      public TrajectoryFrame[] frames; }

    [Tooltip("Assign object_trajectory_unity.json")]
    public TextAsset trajectoryJson;
    public bool loop = true;
    [Tooltip("Matches UniVRM (VRM 1.0) X mirroring. Turn off if left/right looks flipped")]
    public bool mirrorX = true;
    [Tooltip("Playback time offset in seconds; also adjustable with the arrow keys")]
    public float timeOffset = 0f;

    ObjectTrajectory data;
    Transform target;
    float startTime;

    void Start()
    {
        if (trajectoryJson == null)
        { Debug.LogWarning("trajectoryJson is not assigned"); enabled = false; return; }
        data = JsonUtility.FromJson<ObjectTrajectory>(trajectoryJson.text);
        if (data == null || data.frames == null || data.frames.Length == 0)
        { Debug.LogWarning("Trajectory data is empty"); enabled = false; return; }

        PrimitiveType prim = data.shape == "cylinder" ? PrimitiveType.Cylinder
                           : data.shape == "sphere" ? PrimitiveType.Sphere
                           : PrimitiveType.Cube;
        GameObject go = GameObject.CreatePrimitive(prim);
        go.name = "TrackedObject_" + data.label;
        Destroy(go.GetComponent<Collider>());   // no physics: this is kinematic replay
        go.transform.localScale = Vector3.one * data.size;
        var renderer = go.GetComponent<Renderer>();
        renderer.material.color = new Color(0.85f, 0.30f, 0.30f);
        target = go.transform;
        startTime = Time.time;
    }

    void Update()
    {
        if (data == null) return;
        if (Input.GetKeyDown(KeyCode.Space)) startTime = Time.time;
        // Nudge the sync with the arrow keys: Left = earlier, Right = later
        if (Input.GetKeyDown(KeyCode.LeftArrow)) timeOffset += 0.1f;
        if (Input.GetKeyDown(KeyCode.RightArrow)) timeOffset -= 0.1f;

        float duration = data.frames[data.frames.Length - 1].t;
        float t = Time.time - startTime + timeOffset;
        if (t < 0f) t = loop ? t + duration : 0f;
        if (t > duration)
        {
            if (loop) { startTime = Time.time; t = 0f; }
            else t = duration;
        }
        // Interpolate between samples at playback time; the data stays discrete
        int i = 0;
        while (i < data.frames.Length - 2 && data.frames[i + 1].t < t) i++;
        TrajectoryFrame a = data.frames[i];
        TrajectoryFrame b = data.frames[Math.Min(i + 1, data.frames.Length - 1)];
        float u = b.t > a.t ? Mathf.Clamp01((t - a.t) / (b.t - a.t)) : 0f;
        target.position = Vector3.Lerp(ToUnity(a), ToUnity(b), u);
    }

    Vector3 ToUnity(TrajectoryFrame f)
        => new Vector3(mirrorX ? -f.x : f.x, f.y, f.z);

    void OnGUI()
    {
        // Sync status overlay -- delete this method to keep it out of a recording
        GUI.Label(new Rect(10, Screen.height - 44, 500, 20),
            "Object sync: Space=restart  Arrow L/R=offset " +
            timeOffset.ToString("+0.0;-0.0") + "s");
    }
}
'''
    (UNITY_DIR / "ObjectTrajectoryPlayer.cs").write_text(
        OBJECT_PLAYER_CS, encoding="utf-8")
    print("Wrote:", UNITY_DIR / "object_trajectory_unity.json",
          f"({NUM_FRAMES} frames, shape={obj_shape})")
    print("Wrote:", UNITY_DIR / "ObjectTrajectoryPlayer.cs")
else:
    print("No object trajectory: skipping object_trajectory_unity.json")

(UNITY_DIR / "README_UNITY.md").write_text("""# Unity / VRM Output

Avatar model and motion are kept separate:

- `avatar.vrm` ..... place a VRM 1.0 avatar here
  (e.g. VRM Consortium sample `Seed-san.vrm`)
- `motion.vrma` .... VRM Animation exported from Common Behavior Data
- `unity_vrm_animation.mp4` ... capture this locally in Unity

## Recommended: UniVRM SimpleVrma sample (no coding required)

1. Create a Unity project (2022.3 LTS or Unity 6, 3D template)
2. From https://github.com/vrm-c/UniVRM/releases download BOTH
   `UniVRM-0.1xx.x_xxxx.unitypackage` and
   `VRM_Samples-0.1xx.x_xxxx.unitypackage`, then import both via
   `Assets > Import Package > Custom Package...`
3. Open the scene `Assets/VRM10_Samples/SimpleVrma/SimpleVrma` and press Play
4. Use the on-screen UI to open `avatar.vrm` and `motion.vrma`
5. Turn OFF the `BoxMan` checkbox to hide the white skeleton preview
   (if BoxMan moves but the avatar does not, the issue is on the avatar
   side; if neither moves, the issue is on the vrma side)
6. Record the Game view with Unity Recorder
   (`Window > Package Manager` -> install Recorder ->
   `Window > General > Recorder` -> Movie / H.264 MP4)

Recording fps does NOT need to match the analysis fps - panes are
aligned by timestamp, not by frame index. Just cover the full
`duration_sec` listed in `04_behavior_dataset/manifest.json`.

Unity performs no MediaPipe inference, motion smoothing, joint
calculation, or dataset generation - Load VRM, Load VRMA, Play only.

## Optional: replay the object trajectory (Level 1, kinematic)

The object trajectory is part of the Common Behavior Data, so it can be
replayed in Unity just like the human motion:

1. Copy `object_trajectory_unity.json` and `ObjectTrajectoryPlayer.cs`
   into the Unity project's `Assets/` folder
2. In the SimpleVrma scene (NOT in play mode), create an empty GameObject
   and add the `ObjectTrajectoryPlayer` component
3. Assign `object_trajectory_unity.json` to the `trajectoryJson` field
4. Press Play; a primitive matching the object class appears and follows
   the recorded trajectory. Press **Space** to restart the object motion
   at the moment the avatar's motion loops back to the start
5. If left/right looks mirrored against the avatar, turn OFF `mirrorX`

Position depth is monocular-estimated (see `source` per frame); rotation
and physical contact are out of scope for this level.

After capturing, rename the recording to `unity_vrm_animation.mp4` and
copy it back into this folder, beside motion.vrma. A three-screen
comparison built that way ships with the repository in
examples/human-capture/sample_output/05_comparison/.
""", encoding="utf-8")

print("Wrote:", UNITY_DIR / "README_UNITY.md")


In [ ]:
# @title [A5] Adapter report + acceptance check + package
# =====================================================================
# Unity / VRM adapter 5/5.
# =====================================================================
def print_checks(title, checks):
    print(f"=== Acceptance criteria ({title}) ===")
    for label, ok in checks:
        print(("  [x] " if ok else "  [ ] ") + label)


def package_adapter_output(zip_name, entries):
    """Zip this adapter's output with paths relative to the project root.

    `entries` may hold directories or single files. Unzipping the result into
    /content/human_behavior_demo_2_0/ on another runtime puts every file back
    where the other notebooks expect it.
    """
    zip_path = PROJECT_DIR / zip_name
    members = []
    for entry in entries:
        entry = Path(entry)
        if entry.is_dir():
            members += [p for p in sorted(entry.rglob("*")) if p.is_file()]
        elif entry.is_file():
            members.append(entry)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in members:
            archive.write(path, path.relative_to(PROJECT_DIR))
    print(f"\nAdapter output: {zip_path} "
          f"({zip_path.stat().st_size / 1024 / 1024:.1f} MB, "
          f"{len(members)} files)")
    print(f"  colab download -s <session> {zip_path} ./{zip_name}")
    if SHOW_MEDIA:
        try:
            from google.colab import files as colab_files
            if zip_path.stat().st_size < 200 * 1024 * 1024:
                colab_files.download(str(zip_path))
            else:
                print("  (over 200 MB: use the file pane on the left)")
        except Exception:  # noqa: BLE001
            print("  (automatic download unavailable; use the file pane)")
    return zip_path
REPORT = write_adapter_report(
    UNITY_DIR / "adapter_report.json",
    adapter="unity_vrm",
    target="VRM Animation (glTF + VRMC_vrm_animation) for VRM 1.0 avatars",
    reads=["manifest.json", "behavior_summary.json", "timeline/frames.jsonl"],
    unrepresented=[
        "face landmarks and blendshapes: VRM expressions are not written, so "
        "the avatar's face stays neutral",
        "per-joint finger flexion: the mean curl is split evenly across the "
        "three finger segments instead of driving each joint",
        "wrist twist beyond the hand bone rotation",
        "object rotation: only the object's position is replayed",
        "interaction candidates and captions: not baked into the animation; "
        "they stay in the dataset",
        "physical contact: playback is kinematic",
    ])

CHECKS = [
    ("Input: canonical dataset located",
     (DATASET_DIR / "manifest.json").exists()),
    ("Output: motion.vrma exported", VRMA_PATH.exists()),
    ("Output: GLB container is non-trivial",
     VRMA_PATH.exists() and VRMA_PATH.stat().st_size > 1024),
    ("Bones: every canonical humanoid bone has a track",
     all(vrm in rotation_tracks for vrm in COMMON_TO_VRM.values())),
    ("Fingers: exported when a hand was detected",
     EXPORT_FINGERS == bool(HAND_PRESENT["Left"].any()
                            or HAND_PRESENT["Right"].any())),
    ("Object: trajectory + player script for Unity",
     (UNITY_DIR / "object_trajectory_unity.json").exists()
     and (UNITY_DIR / "ObjectTrajectoryPlayer.cs").exists()),
    ("Dataset view: adapters/vrm_bone_rotations.csv",
     (DATASET_DIR / "adapters/vrm_bone_rotations.csv").exists()),
    ("Docs: README_UNITY.md", (UNITY_DIR / "README_UNITY.md").exists()),
    ("Output: adapter_report.json written",
     (UNITY_DIR / "adapter_report.json").exists()),
    ("Unity: unity_vrm_animation.mp4 (captured locally, optional)",
     (UNITY_DIR / "unity_vrm_animation.mp4").exists()),
]
print_checks("unity / vrm adapter", CHECKS)

package_adapter_output("vrm_adapter_output.zip", [
    UNITY_DIR,
    DATASET_DIR / "adapters/vrm_bone_rotations.csv",
])
print("\nNext: play motion.vrma in Unity (guide below), record the Game view, "
      "and drop the recording back in as "
      "output/03_unity_vrm/unity_vrm_animation.mp4 -- that is the third pane of "
      "a comparison video (example: examples/human-capture/sample_output/"
      "05_comparison/).")


## Play it on a VRM avatar in Unity (outside Colab, optional)

Unity is **playback only** here — no inference, no motion computation. With
UniVRM's **SimpleVrma** sample scene you do not have to write a line of code.

1. Create a 3D project in Unity 2022.3 LTS (or Unity 6)
2. From [UniVRM Releases](https://github.com/vrm-c/UniVRM/releases) download
   **both** `UniVRM-0.1xx.x_xxxx.unitypackage` and
   `VRM_Samples-0.1xx.x_xxxx.unitypackage`, then import them via
   `Assets > Import Package > Custom Package...`
3. Open the scene `Assets/VRM10_Samples/SimpleVrma/SimpleVrma` and press ▶ Play
4. Load your avatar (`avatar.vrm`, e.g. the VRM Consortium's Seed-san) and the
   `motion.vrma` this notebook produced
5. Turn **BoxMan** off to hide the white skeleton preview.
   (Diagnostic: if BoxMan moves but the avatar does not, the problem is on the
   avatar side; if neither moves, it is on the vrma side.)
6. Record the Game view with Unity Recorder — install it from Package Manager,
   then `Window > General > Recorder`. **Recording at 30/60 fps is fine**:
   panes are aligned by timestamp, not by frame index. Just cover the
   `duration_sec` listed in `manifest.json`.
7. Rename the recording to `unity_vrm_animation.mp4` and put it back into
   `output/03_unity_vrm/`, so it sits beside `motion.vrma`:

```bash
colab upload -s cbd ./unity_vrm_animation.mp4 \
  /content/human_behavior_demo_2_0/output/03_unity_vrm/unity_vrm_animation.mp4
```

Because this step needs Unity on your own machine, it is outside the CLI
chain. A three-screen comparison built this way ships with the repository:
[`examples/human-capture/sample_output/05_comparison/`](../../examples/human-capture/sample_output/05_comparison/).

Optionally replay the object alongside the avatar with
`object_trajectory_unity.json` + `ObjectTrajectoryPlayer.cs` — step by step in
`output/03_unity_vrm/README_UNITY.md`, written by `[A4]`.
